In [ ]:
!pip install supabase

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

WORK_DIR = '/content/drive/MyDrive/Proyecto_RAG_Coyuntura'
DATA_DIR = os.path.join(WORK_DIR, 'datos')
os.makedirs(DATA_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from google.colab import userdata
os.environ["SUPABASE_URL"] = userdata.get("SUPABASE_URL")
os.environ["SUPABASE_KEY"] = userdata.get("SUPABASE_KEY")

url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_KEY")

In [ ]:
import pandas as pd
import numpy as np
from supabase import create_client, Client

supabase = create_client(url, key)

response = (
    supabase.table("urgente_articles")
    .select("*")
    .range(0, 9999)
    .execute()
)

data = response.data
df_db = pd.DataFrame(data)

print("Rows retrieved:", len(df_db))

Rows retrieved: 3696


In [ ]:
# Limpieza de datos
columnas = ["date", "title", "url", "body_text"]

df_noticias = df_db[columnas].copy()
df_noticias = df_noticias.dropna() # Filtrado de noticias con datos faltantes
def limpiar_cuerpo(texto):
    partes = texto.split(").-", 1)  # divide como máximo en 2 partes
    cuerpo = partes[1] if len(partes) > 1 else partes[0]
    return " ".join(cuerpo.split())  # normaliza espacios/saltos de línea

df_noticias["body_text"] = df_noticias["body_text"].astype(str).apply(limpiar_cuerpo)
df_noticias["date"] = pd.to_datetime(df_noticias["date"]) # Correcion de formato de fecha
df_noticias.columns = ["Fecha", "Título", "Url", "Cuerpo"] # Renombramiento de columnas

df_noticias

,Fecha,Título,Url,Cuerpo
0,2026-01-21,“Algunos no trabajan”: Legisladores se critica...,https://www.urgente.bo/noticia/%E2%80%9Calguno...,"Este martes, el senador suplente de Alianza Un..."
1,2026-01-20,“Era inviable para el 2026”: El “50-50” carecí...,https://www.urgente.bo/noticia/%E2%80%9Cera-in...,"Este lunes, el presidente Rodrigo Paz afirmó q..."
2,2026-01-20,“La inversión privada debe generar movilidad s...,https://www.urgente.bo/noticia/%E2%80%9Cla-inv...,"José Gabriel Espinoza, ministro de Economía, a..."
3,2026-01-23,Acribillan a un hombre dentro de su habitación...,https://www.urgente.bo/noticia/acribillan-un-h...,Un hombre fue encontrado sin vida este viernes...
4,2026-01-20,Actores políticos cuestionan a Paz por retrasa...,https://www.urgente.bo/noticia/actores-pol%C3%...,"Este lunes, el presidente Rodrigo Paz afirmó q..."
...,...,...,...,...
3691,2026-07-15,Paz dice que los bloqueos fueron promovidos po...,https://www.urgente.bo/noticia/paz-dice-que-lo...,El presidente Rodrigo Paz afirmó este miércole...
3692,2026-07-15,Reclutamiento de bolivianos para la guerra con...,https://www.urgente.bo/noticia/reclutamiento-d...,15 de julio (Urgente.bo)- Los informes acerca ...
3693,2026-07-15,Revilla convoca a un gran acuerdo para impulsa...,https://www.urgente.bo/noticia/revilla-convoca...,"Este miércoles, en la Sesión de Honor de la As..."
3694,2026-07-16,"Si Bolivia importa gas natural, que es cada ve...",https://www.urgente.bo/noticia/si-bolivia-impo...,Bolivia se enfrenta a un escenario cada vez má...


In [ ]:
# Guardado del dataset preparado
df_noticias.to_csv(os.path.join(DATA_DIR, "noticias.csv"), index=False)
print("Dataset de noticias guardado con exito!")

Dataset de noticias guardado con exito!
